# 03 — Error Analysis

FN/FP experiment time-series drill-down using OOF predictions.

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import plotly.express as px
from src.config import OOF_DIR, REPORTS_DIR
from src.evaluate import compute_metrics
from src.load_data import load_train, load_experiment_timeseries

log = pd.read_csv(REPORTS_DIR / 'experiment_log.csv')
best = log[(log.fold=='oof')].sort_values(['recall','f1'], ascending=False).iloc[0]
meta = json.loads((OOF_DIR / f"{best.run_id}_meta.json").read_text())
probs = np.load(OOF_DIR / f"{best.run_id}_oof.npy")
y = load_train().set_index('experiment_id').loc[meta['experiment_ids'], 'label'].values
thr = float(best.threshold or 0.5)
print(compute_metrics(y, probs, thr))

In [ ]:
pred = (probs >= thr).astype(int)
fn_ids = [meta['experiment_ids'][i] for i in range(len(y)) if y[i]==1 and pred[i]==0]
fp_ids = [meta['experiment_ids'][i] for i in range(len(y)) if y[i]==0 and pred[i]==1]
print('FN', fn_ids, 'FP', fp_ids)
if fn_ids:
    ts = load_experiment_timeseries(int(fn_ids[0]))
    px.line(ts, y='S1_CurrentFeedback', title=f'FN experiment_{fn_ids[0]:02d}').show()